# Milestone 1 Exploration

The goal of this notebook is to explore a small subset of the Amazon Movies and TV review and metadata files, understand the available fields, and justify the field selection and preprocessing decisions for retrieval.

In [39]:
import gzip
import json
import pandas as pd

review_path = "../data/raw/Movies_and_TV.jsonl.gz"
meta_path = "../data/raw/meta_Movies_and_TV.jsonl.gz"

meta_lookup = {}

with gzip.open(meta_path, "rt", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        parent_asin = str(record.get("parent_asin", "")).strip()

        raw_title = record.get("title", "")
        title = ""
        if raw_title is not None:
            title = str(raw_title).strip()

        if parent_asin and title:
            meta_lookup[parent_asin] = record

print("Metadata products with titles:", len(meta_lookup))

Metadata products with titles: 434222


In [40]:
matched_reviews = []
matched_asins = set()

with gzip.open(review_path, "rt", encoding="utf-8") as f:
    for line in f:
        review = json.loads(line)
        parent_asin = str(review.get("parent_asin", "")).strip()

        if parent_asin in meta_lookup:
            matched_reviews.append(review)
            matched_asins.add(parent_asin)

            if len(matched_reviews) == 200:
                break

reviews_sample = pd.DataFrame(matched_reviews)

In [41]:
matched_meta = [meta_lookup[asin] for asin in matched_asins]
meta_sample = pd.DataFrame(matched_meta)

print("Reviews sample shape:", reviews_sample.shape)
print("Metadata sample shape:", meta_sample.shape)

reviews_sample.head()
meta_sample.head()

Reviews sample shape: (200, 10)
Metadata sample shape: (199, 15)


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle
0,Movies & TV,The Polar Express (Widescreen Edition) [DVD],4.8,90710,[],"[Product Description, Polar Express (DVD) (WS)...",7.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],"Nona Gaye (Actor), Peter Scolari (Actor)...","[Movies & TV, Today's Deals, Featured Deals & ...","{'Genre': 'Kids & Family, Animation, Science F...",B000AGTPUK,None,NaN
1,Movies & TV,A Christmas Carol,4.8,9932,[],[Alastair Sim's tour-de-force performance as t...,9.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],"Alastair Sim (Actor), Hermione Baddely (...","[Movies & TV, Featured Categories, DVD, Drama]","{'Aspect Ratio': '1.33:1', 'Is Discontinued By...",B008UY8FI2,None,NaN
2,Movies & TV,Tangled (Four-Disc Combo: Blu-ray 3D / Blu-ray...,4.8,23523,[],[Disney presents a new twist on one of the mos...,39.87,[{'thumb': 'https://m.media-amazon.com/images/...,[],"Mandy Moore (Actor), Zachary Levi (Actor...","[Movies & TV, Featured Categories, Blu-ray, Ki...","{'Genre': 'Kids & Family, Action & Adventure',...",B004G6009K,None,NaN
3,Movies & TV,Spartacus: Vengeance: Season 2,4.8,3426,[],[10-episode sequel to “Spartacus: Blood & Sand...,14.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Trailer for Spartacus', 'url': 'ht...","Liam McIntyre (Actor), Katrina Law (Acto...","[Movies & TV, Featured Categories, Blu-ray, Dr...","{'Aspect Ratio': '1.78:1', 'Is Discontinued By...",B0072OFOM6,None,NaN
4,Movies & TV,Hugo (Blu-ray 3D + Blu-ray + DVD + Digital Cop...,4.7,6549,[],"[Product Description, Welcome to a magical wor...",40.62,[{'thumb': 'https://m.media-amazon.com/images/...,[],"Asa Butterfield (Actor), Chloë Grace More...","[Movies & TV, Blu-ray, Movies]","{'Genre': 'Action & Adventure', 'Format': 'AC-...",B006OAXL92,None,NaN


## Dataset overview

The first 200 entries are used from both the review file and the metadata file for quick exploration. This is enough to inspect the structure of the data and decide which fields are useful for retrieval, without loading the full dataset into memory.

In [20]:
print("Reviews sample shape:", reviews_sample.shape)
print("Metadata sample shape:", meta_sample.shape)

print("\nReview columns:")
print(reviews_sample.columns.tolist())

print("\nMetadata columns:")
print(meta_sample.columns.tolist())

Reviews sample shape: (200, 10)
Metadata sample shape: (200, 15)

Review columns:
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

Metadata columns:
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle']


In [21]:
reviews_sample.info()
meta_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   rating             200 non-null    float64
 1   title              200 non-null    str    
 2   text               200 non-null    str    
 3   images             200 non-null    object 
 4   asin               200 non-null    str    
 5   parent_asin        200 non-null    str    
 6   user_id            200 non-null    str    
 7   timestamp          200 non-null    int64  
 8   helpful_vote       200 non-null    int64  
 9   verified_purchase  200 non-null    bool   
dtypes: bool(1), float64(1), int64(2), object(1), str(5)
memory usage: 14.4+ KB
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    197 non-null    str    
 1   title        

## Inspection of Sample Records

In [22]:
reviews_sample[["title", "text", "rating", "parent_asin"]].head(5)

,title,text,rating,parent_asin
0,Five Stars,"Amazon, please buy the show! I'm hooked!",5.0,B013488XFS
1,Five Stars,My Kiddos LOVE this show!!,5.0,B00CB6VTDS
2,Some decent moments...but...,Annabella Sciorra did her character justice wi...,3.0,B096Z8Z3R6
3,"Decent Depiction of Lower-Functioning Autism, ...",...there should be more of a range of characte...,4.0,B09M14D9FZ
4,What Love Is...,"...isn't always how you expect it to be, but w...",5.0,B001H1SVZC


In [23]:
meta_sample[["title", "description", "features", "categories", "parent_asin"]].head(5)

,title,description,features,categories,parent_asin
0,The Wiz,[Ease on down the yellow-brick road with The W...,[],"[Movies & TV, Music Artists, Jackson, Michael]",B001JV5BF8
1,On The Case with Paula Zahn,[On The Case with Paula Zahn takes viewers on ...,"[2018, 19 episodes, 13+]","[Suspense, Drama, Special Interest]",B07FLP19K1
2,Cinderella (Two-Disc Diamond Edition Blu-ray/D...,"[Product Description, Rediscover the timeless ...",[],"[Movies & TV, Blu-ray, Movies]",B007WWRJA8
3,Watchmen (Director's Cut),"[Product Description, Watchmen: Special Editio...",[],"[Movies & TV, Genre for Featured Categories, A...",B001QTXM5Y
4,Seven Girlfriends,[Jesse has never had a problem finding a girl...,"[IMDb 5.7, 1 h 32 min, 2000, R]","[Comedy, Romance, Charming, Intense]",B01KBKKU7G


In [24]:
type(meta_sample.loc[0, "description"]), meta_sample.loc[0, "description"]

(list,
 ["Ease on down the yellow-brick road with The Wiz, starring superstars Diana Ross and Michael Jackson! Relive all of the magic of this beloved musical when Dorothy is whisked away to the enchanting wonderland of Oz, where she encounters the Scarecrow, the Tin Man and the Lion. The Wiz features spectacular musical numbers from legendary producer Quincy Jones and an all-star cast including Lena Horne, Richard Pryor, Nipsey Russell and Ted Ross. With a digitally remastered picture and the unforgettable soundtrack in new 5.1 surround sound, it's an experience of the land of Oz unlike anything you've seen before!Bonus Content:",
  'Wiz on Down the Road',
  'Wiz on Down the Road',
  'Trailer',
  'Trailer'])

In [25]:
type(meta_sample.loc[0, "features"]), meta_sample.loc[0, "features"]

(list, [])

In [26]:
type(meta_sample.loc[0, "categories"]), meta_sample.loc[0, "categories"]

(list, ['Movies & TV', 'Music Artists', 'Jackson, Michael'])

In [27]:
type(reviews_sample.loc[0, "images"]), reviews_sample.loc[0, "images"]

(list, [])

## Field selection for retrieval

Based on the sample records, different fields are selected from the review and metadata files depending on whether they contain useful natural language text.

For the review data, the most useful fields for retrieval are `title` and `text`. These two fields contain the main user-written content and are the most likely to match search queries effectively. `rating` is kept as metadata because it may still be useful later for filtering or analysis, and `parent_asin` is kept so that reviews can be linked to the corresponding product metadata. Fields such as `user_id` and `timestamp` are not used as main retrieval text because they are identifiers or administrative information rather than descriptive content. The `images` field is also not selected because it is a list and appears empty in the sample records.

For the metadata file, the most useful fields for retrieval are `title`, `description`, `features`, and `categories`. These fields describe what the product is and contain text that could help both keyword-based and semantic retrieval. Since `description`, `features`, and `categories` are stored as lists, they need to be flattened into text during preprocessing. `parent_asin` is also kept as metadata so that metadata records can be joined with reviews. Fields such as `price`, `average_rating`, and `rating_number` may still be useful as metadata, but they are not the main retrieval text. Fields like `subtitle` and `bought_together` are not prioritized because they are sparse in this sample.

## Preprocessing decisions

For preprocessing, only fields that are useful for retrieval or record linkage are kept. In the review data, `title` and `text` are combined into a single retrieval text field so that both the short review summary and the full review body can be searched together. In the metadata data, `title`, `description`, `features`, and `categories` are combined into one text field. Because several metadata fields are stored as lists, they are first converted into plain text by joining their elements.

`parent_asin` is kept as metadata so that review records and metadata records can be connected later. At this stage, preprocessing should stay fairly light. Text can be lowercased, stripped of extra whitespace, and empty retrieval fields can be removed, but overly aggressive cleaning is avoided since wording and punctuation may still be useful for retrieval. Fields such as `user_id`, `timestamp`, and image-related fields are excluded from the main retrieval text because they do not add meaningful semantic content.

In [28]:
reviews_processed = reviews_sample[["parent_asin", "title", "text", "rating"]].copy()

reviews_processed["title"] = reviews_processed["title"].fillna("").str.strip()
reviews_processed["text"] = reviews_processed["text"].fillna("").str.strip()

reviews_processed["retrieval_text"] = (
    reviews_processed["title"] + ". " + reviews_processed["text"]
).str.strip()

reviews_processed = reviews_processed[reviews_processed["retrieval_text"].str.len() > 0]

reviews_processed.head()

,parent_asin,title,text,rating,retrieval_text
0,B013488XFS,Five Stars,"Amazon, please buy the show! I'm hooked!",5.0,"Five Stars. Amazon, please buy the show! I'm h..."
1,B00CB6VTDS,Five Stars,My Kiddos LOVE this show!!,5.0,Five Stars. My Kiddos LOVE this show!!
2,B096Z8Z3R6,Some decent moments...but...,Annabella Sciorra did her character justice wi...,3.0,Some decent moments...but.... Annabella Sciorr...
3,B09M14D9FZ,"Decent Depiction of Lower-Functioning Autism, ...",...there should be more of a range of characte...,4.0,"Decent Depiction of Lower-Functioning Autism, ..."
4,B001H1SVZC,What Love Is...,"...isn't always how you expect it to be, but w...",5.0,What Love Is.... ...isn't always how you expec...


In [29]:
meta_processed = meta_sample[["parent_asin", "title", "description", "features", "categories"]].copy()

meta_processed["title"] = meta_processed["title"].fillna("").str.strip()

meta_processed["description"] = meta_processed["description"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
)

meta_processed["features"] = meta_processed["features"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
)

meta_processed["categories"] = meta_processed["categories"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
)

meta_processed["retrieval_text"] = (
    meta_processed["title"] + ". "
    + meta_processed["description"] + " "
    + meta_processed["features"] + " "
    + meta_processed["categories"]
).str.strip()

meta_processed = meta_processed[meta_processed["retrieval_text"].str.len() > 0]

meta_processed.head()

,parent_asin,title,description,features,categories,retrieval_text
0,B001JV5BF8,The Wiz,Ease on down the yellow-brick road with The Wi...,,"Movies & TV Music Artists Jackson, Michael",The Wiz. Ease on down the yellow-brick road wi...
1,B07FLP19K1,On The Case with Paula Zahn,On The Case with Paula Zahn takes viewers on a...,2018 19 episodes 13+,Suspense Drama Special Interest,On The Case with Paula Zahn. On The Case with ...
2,B007WWRJA8,Cinderella (Two-Disc Diamond Edition Blu-ray/D...,Product Description Rediscover the timeless st...,,Movies & TV Blu-ray Movies,Cinderella (Two-Disc Diamond Edition Blu-ray/D...
3,B001QTXM5Y,Watchmen (Director's Cut),Product Description Watchmen: Special Edition ...,,Movies & TV Genre for Featured Categories Acti...,Watchmen (Director's Cut). Product Description...
4,B01KBKKU7G,Seven Girlfriends,Jesse has never had a problem finding a girlf...,IMDb 5.7 1 h 32 min 2000 R,Comedy Romance Charming Intense,Seven Girlfriends. Jesse has never had a prob...


In [30]:
reviews_processed.to_csv("../data/processed/reviews_sample_processed.csv", index=False)
meta_processed.to_csv("../data/processed/meta_sample_processed.csv", index=False)

In [31]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from src.bm25 import BM25Retriever

meta_df = pd.read_csv('../data/processed/meta_sample_processed.csv')
reviews_df = pd.read_csv('../data/processed/reviews_sample_processed.csv')

In [32]:
merged_df = pd.merge(
    reviews_df, 
    meta_df[['parent_asin', 'title']], 
    on='parent_asin', 
    how='left', 
    suffixes=('_review', '_product')
)

merged_df['title_product'] = merged_df['title_product'].fillna("Title Unavailable in Sample")

In [33]:
corpus_records = []
for _, row in merged_df.iterrows():
    record = {
        'parent_asin': row['parent_asin'],
        'retrieval_text': str(row['retrieval_text']), # Ensure it's a string
        'product_title': str(row['title_product']),   
        'review_text': str(row['text_review'] if 'text_review' in row else row['text']),              
        'rating': row['rating']                  
    }
    corpus_records.append(record)

In [34]:
retriever = BM25Retriever(
    index_path='../data/processed/bm25_index.pkl',
    corpus_path='../data/processed/corpus_data.pkl'
)

retriever.build_and_save_index(corpus_records)

Tokenizing corpus...
Building BM25 Index...
Index successfully saved to ../data/processed/bm25_index.pkl


In [35]:
retriever.load_index()
results = retriever.search("great software", top_n=1)

print("\n--- TEST SEARCH RESULTS ---")
for doc, score in results:
    print(f"Score: {score:.2f}")
    print(f"Product: {doc['product_title']}")
    print(f"Rating: {doc['rating']} Stars")
    print(f"Review: {doc['review_text'][:100]}...")


--- TEST SEARCH RESULTS ---
Score: 2.59
Product: Days Of Thunder [Blu-ray]
Rating: 5.0 Stars
Review: Gift for son.  Great price for blue ray....
